In [0]:

# CREAR ESQUEMA GOLD

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS rendimiento_estudiantil.gold
""")

Leer tablas silver

In [0]:
silver_estudiantes = spark.table(f"rendimiento_estudiantil.silver.estudiantes")

silver_cursos = spark.table(f"rendimiento_estudiantil.silver.cursos")

silver_matriculas = spark.table(f"rendimiento_estudiantil.silver.matriculas")

silver_semestres = spark.table(f"rendimiento_estudiantil.silver.semestres")

## Construcción de la Dimensión Estudiantes

Esta dimensión almacena la información descriptiva de cada estudiante.

In [0]:
dim_estudiantes = (
    silver_estudiantes
    .select(
        "Student_ID",
        "Age",
        "Gender",
        "Region_Type",
        "Family_Size",
        "Parent_Education",
        "Family_Income_Level",
        "Home_City",
        "Major_Subject",
        "email"
    )
)

In [0]:
(
    dim_estudiantes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"rendimiento_estudiantil.gold.dim_estudiantes")
)

## Construcción de la Dimensión Cursos

Contiene la información académica de cada curso.

In [0]:
dim_cursos = (
    silver_cursos
    .select(
        "id_curso",
        "nombre_curso",
        "major_subject",
        "creditos",
        "costo_matricula",
        "nivel"
    )
)

In [0]:
(
    dim_cursos.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"rendimiento_estudiantil.gold.dim_cursos")
)

## Construcción de la Dimensión Semestres

Esta dimensión representa el calendario académico.

In [0]:
dim_semestres = (
    silver_semestres
    .select(
        "Semester_ID",
        "periodo",
        "anio",
        "ciclo",
        "modalidad"
    )
)

In [0]:
(
    dim_semestres.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"rendimiento_estudiantil.gold.dim_semestres")
)

## Construcción de la Tabla de Hechos

La tabla de hechos registra cada matrícula realizada por un estudiante y contiene las métricas principales para el análisis.

Realizamos los joins

In [0]:
# Renombrar costo_matricula de dim_cursos para evitar ambigüedad
dim_cursos_renamed = dim_cursos.withColumnRenamed("costo_matricula", "costo_curso_estandar")

fact_matriculas = (
    silver_matriculas
        .join(
            dim_estudiantes,
            "Student_ID"
        )
        .join(
            dim_cursos_renamed,
            "id_curso"
        )
        .join(
            dim_semestres,
            "Semester_ID"
        )
)

Seleccionamos unicamente las columnas necesarias

In [0]:
fact_matriculas = (
    fact_matriculas.select(
        "id_matricula",
        "Student_ID",
        "id_curso",
        "Semester_ID",
        "fecha_matricula",
        "nota_final",
        "costo_matricula",           
        "costo_curso_estandar",     
        "estado"
    )
)

In [0]:
(
    fact_matriculas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"rendimiento_estudiantil.gold.fact_matriculas")
)

Verifificamos

In [0]:
%sql
SHOW TABLES IN rendimiento_estudiantil.gold

Ver el modelo

In [0]:
display(dim_estudiantes)

display(dim_cursos)

display(dim_semestres)

display(fact_matriculas)